# 📊 **Sesión 30 — Preparación para Machine Learning: variables, datos y validación**
# Diplomado en Ciencia de Datos · jueves 20 de agosto de 2026




---
## **¿Qué es Machine Learning? (el puente entre estadística e IA)**

Ya llevamos varias sesiones trabajando con estadística: regresión lineal, modelos lineales generalizados, estadística bayesiana, series de tiempo.

**La idea central: cuando la estadística pasa la línea y pasamos a la IA**

> Todo lo que hicimos con regresión lineal — ajustar una recta a unos datos para predecir un valor — **ya era Machine Learning**, aunque no lo llamáramos así.

La diferencia no está tanto en las matemáticas (mucho es lo mismo), sino en el **enfoque**:

- La **Estadística Clásica** parte de supuestos sobre cómo se distribuyen los datos y busca *explicar e inferir* con rigor matemático.
- El **Machine Learning** pone el foco en que el modelo *aprenda de los datos* y **generalice bien a datos que nunca ha visto**, aunque no entendamos perfectamente por qué funciona.

Por eso en la sesión de hoy vamos a enfocarnos en entender **las reglas del juego que todo modelo de Machine Learning debe respetar**, sea el que sea:

1.   Cómo preparamos las variables
2.   Cómo dividimos los datos
3.   Cómo sabemos si el modelo realmente aprendió o solo memorizó.


In [ ]:
# Celda de preparación — ejecútala primero, es la única "técnica" del notebook.
# Importa las herramientas que usaremos. No hace falta instalar nada: ya vienen en Google Colab.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.datasets import make_classification

np.random.seed(42)  # fija la "semilla" aleatoria para que todos obtengamos los mismos resultados

print("✅ Todo listo. Las herramientas están cargadas y podemos continuar.")

✅ Todo listo. Las herramientas están cargadas y podemos continuar.


---
## 1️⃣ **Bloque 1 — Tipos de variables**

Vamos a crear un dataset sencillo de **precios de viviendas** con varias columnas para poder hablar de los distintos tipos de variables que existen.


In [ ]:
n = 300

viviendas = pd.DataFrame({
    "superficie_m2": np.random.normal(90, 25, n).round(1),
    "n_habitaciones": np.random.choice([1, 2, 3, 4, 5], n),
    "barrio": np.random.choice(["Centro", "Norte", "Sur", "Este"], n),
    "antiguedad_años": np.random.randint(0, 50, n),
    "tiene_ascensor": np.random.choice(["Sí", "No"], n),
})

# El precio depende (con algo de ruido) de la superficie, habitaciones y antigüedad
viviendas["precio_clp"] = (
    2_200_000 * viviendas["superficie_m2"]
    + 8_000_000 * viviendas["n_habitaciones"]
    - 900_000 * viviendas["antiguedad_años"]
    + np.random.normal(0, 15_000_000, n)
).round(0)

viviendas.head()

,superficie_m2,n_habitaciones,barrio,antiguedad_años,tiene_ascensor,precio_clp
0,102.4,1,Norte,32,No,242005473.0
1,86.5,4,Centro,36,Sí,196944312.0
2,106.2,2,Sur,26,Sí,200754892.0
3,128.1,5,Centro,32,No,295680631.0
4,84.1,1,Norte,3,Sí,162857769.0


In [ ]:
print("Así se ven los TIPOS de cada variable en nuestra tabla:\n")
print(viviendas.dtypes)

print("""
🔎 Cómo leerlo:
- superficie_m2, antiguedad_años, precio_clp  → variables NUMÉRICAS (números que se pueden sumar/promediar)
- n_habitaciones                                → numérica, pero DISCRETA (solo toma valores enteros: 1,2,3...)
- barrio, tiene_ascensor                        → variables CATEGÓRICAS (categorías, no números)
   - "barrio" es categórica NOMINAL (no hay un orden natural entre Centro/Norte/Sur/Este)
   - si tuviéramos algo como "básico/medio/alto" sería categórica ORDINAL (sí hay orden)
""")

Así se ven los TIPOS de cada variable en nuestra tabla:

superficie_m2      float64
n_habitaciones       int64
barrio              object
antiguedad_años      int64
tiene_ascensor      object
precio_clp         float64
dtype: object

🔎 Cómo leerlo:
- superficie_m2, antiguedad_años, precio_clp  → variables NUMÉRICAS (números que se pueden sumar/promediar)
- n_habitaciones                                → numérica, pero DISCRETA (solo toma valores enteros: 1,2,3...)
- barrio, tiene_ascensor                        → variables CATEGÓRICAS (categorías, no números)
   - "barrio" es categórica NOMINAL (no hay un orden natural entre Centro/Norte/Sur/Este)
   - si tuviéramos algo como "básico/medio/alto" sería categórica ORDINAL (sí hay orden)



**Por qué importa esto:** los modelos de Machine Learning solo entienden números. Antes de entrenar cualquier modelo hay que **convertir** las variables categóricas (por ejemplo, con `pd.get_dummies`, técnica llamada *one-hot encoding*). Hoy no lo veremos, pero basta con que sepamos que ese paso existe y por qué será necesario en el futuro cuando hagamos modelos de Machine Learning.

---
## 2️⃣ **Bloque 2 — Calidad de los datos**

En la vida real, los datos casi nunca llegan perfectos. Vamos a simular un problema típico: **valores faltantes**.


In [ ]:
# Simulamos que a algunas viviendas les falta el dato de antigüedad (algo MUY común en la vida real)
viviendas_sucias = viviendas.copy()
indices_faltantes = np.random.choice(viviendas_sucias.index, size=20, replace=False)
viviendas_sucias.loc[indices_faltantes, "antiguedad_años"] = np.nan

print("¿Cuántos datos faltan por columna?\n")
print(viviendas_sucias.isna().sum())

print(f"\n🔎 Un {20/n:.1%} de nuestras viviendas no tienen dato de antigüedad.")
print("Esto es exactamente el tipo de problema de COMPLETITUD que vimos la sesión pasada:")
print("si ignoramos estos datos sin más, podríamos perder información valiosa o introducir sesgo.")


¿Cuántos datos faltan por columna?

superficie_m2       0
n_habitaciones      0
barrio              0
antiguedad_años    20
tiene_ascensor      0
precio_clp          0
dtype: int64

🔎 Un 6.7% de nuestras viviendas no tienen dato de antigüedad.
Esto es exactamente el tipo de problema de COMPLETITUD que vimos la sesión pasada:
si ignoramos estos datos sin más, podríamos perder información valiosa o introducir sesgo.


**Las 4 dimensiones de calidad de datos** (recordatorio rápido, ya lo vimos):
- **Exactitud**: ¿el dato refleja la realidad?
- **Completitud**: ¿faltan datos? (justo lo que acabamos de ver)
- **Coherencia**: ¿hay contradicciones entre sistemas?
- **Frescura**: ¿está actualizado?


---
## **3️⃣ Bloque 3 — División de datos: entrenamiento, validación y prueba**

Esta es **una de las ideas más importantes de la sesión y se tiene que tener en cuenta siempre que hablemos de entrenamiento de modelos en Machine Learning**.

Antes de entrenar un modelo, separamos los datos para que cada conjunto cumpla un rol distinto. **En este ejemplo usaremos una división 70% / 15% / 15%**, pero estas proporciones pueden cambiar según el tamaño del dataset y la naturaleza del problema:

- **Entrenamiento (70%)**: con estos datos el modelo aprende sus parámetros.
- **Validación (15%)**: se usa durante el desarrollo para comparar alternativas y ajustar el modelo, sin tocar el conjunto de prueba.
- **Prueba (15%)**: se reserva para la evaluación final con datos que no participaron en el entrenamiento ni en los ajustes. Por eso decimos que es "sagrado".

¿Por qué hacemos esta separación? Porque queremos saber si el modelo **generaliza a datos nuevos** y detectar problemas como el **sobreajuste (overfitting)**: cuando un modelo aprende demasiado bien las particularidades o el ruido de los datos de entrenamiento y luego rinde peor con observaciones nuevas.

In [ ]:
X = viviendas[["superficie_m2", "n_habitaciones", "antiguedad_años"]]
y = viviendas["precio_clp"]

# Primera división: separamos el 70% para entrenar y dejamos un 30% "aparte"
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

# Ese 30% lo partimos a la mitad: 15% validación, 15% prueba
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f"✅ Total de viviendas: {len(X)}")
print(f"   Entrenamiento: {len(X_train)} viviendas ({len(X_train)/len(X):.0%})")
print(f"   Validación:    {len(X_val)} viviendas ({len(X_val)/len(X):.0%})")
print(f"   Prueba:        {len(X_test)} viviendas ({len(X_test)/len(X):.0%})  ← esta NO se toca hasta el final")

✅ Total de viviendas: 300
   Entrenamiento: 210 viviendas (70%)
   Validación:    45 viviendas (15%)
   Prueba:        45 viviendas (15%)  ← esta NO se toca hasta el final


---
## **4️⃣ Bloque 4 — Regularización: antes y después de estandarizar**

En este bloque vamos a comparar **el mismo problema de dos maneras**.

### **4A. Primero, sin estandarizar**
Aplicaremos Ridge y Lasso directamente sobre variables medidas en escalas distintas.

Por ejemplo:
- la **superficie** puede tomar valores del orden de cientos;
- el **número de habitaciones** puede tomar valores entre 1 y 6;
- la **antigüedad** puede tomar valores de varias decenas de años.

Si aplicamos una penalización directamente sobre coeficientes asociados a escalas tan distintas, **comparar sus magnitudes puede resultar engañoso**.

### **4B. Después, estandarizando**
Repetiremos exactamente el mismo ejercicio, pero primero transformaremos las variables para que queden en una escala comparable.

Esto nos permitirá observar con más claridad:
- cómo **Ridge (L2)** reduce la magnitud de los coeficientes;
- cómo **Lasso (L1)** puede llevar algunos coeficientes exactamente a cero;
- y por qué la estandarización es especialmente importante cuando usamos regularización.

> **Idea clave:** estandarizar no significa “arreglar un modelo malo”. Significa poner las variables en una escala comparable para que la penalización de Ridge y Lasso sea más interpretable y equilibrada.

Además agregaremos una variable de **ruido aleatorio**, que no participa en la construcción del precio. Así tendremos una variable que, en principio, el modelo debería considerar poco útil.

Piensen en la regularización como **hacer las maletas para un viaje**:
- **Ridge** te permite llevar todo, pero controla cuánto pesa cada cosa.
- **Lasso** puede obligarte a dejar fuera lo que no aporta.

In [ ]:
# ============================================================
# 4A. REGULARIZACIÓN SIN ESTANDARIZAR
# ============================================================

X_reg = X.copy()
rng = np.random.default_rng(42)
X_reg["ruido_aleatorio"] = rng.normal(0, 1, len(X_reg))

# Expresamos el precio en millones de CLP para facilitar la lectura.
y_reg = y / 1_000_000

X_reg_train, X_reg_temp, y_reg_train, y_reg_temp = train_test_split(
    X_reg, y_reg, test_size=0.30, random_state=42
)
X_reg_val, X_reg_test, y_reg_val, y_reg_test = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.50, random_state=42
)

modelos_sin_escala = {
    "Sin regularizar": LinearRegression(),
    "Ridge (L2)": Ridge(alpha=10),
    "Lasso (L1)": Lasso(alpha=2, max_iter=10000),
}

coef_sin_escala = pd.DataFrame(index=X_reg.columns)

for nombre, modelo in modelos_sin_escala.items():
    modelo.fit(X_reg_train, y_reg_train)
    coef_sin_escala[nombre] = modelo.coef_.round(3)

print("4A. Coeficientes SIN estandarizar:\n")
display(coef_sin_escala)

print("""
🧭 ¿Qué mirar aquí?

1. Los coeficientes NO están en una escala común.
   Por eso, un coeficiente numéricamente grande no significa necesariamente
   que esa variable sea "más importante".

2. Ridge y Lasso ya están aplicando una penalización, pero su efecto puede ser
   difícil de comparar porque cada variable está medida en unidades distintas.

3. Mira especialmente la fila 'ruido_aleatorio'.
   Puede tener un coeficiente pequeño, pero todavía no estamos en la mejor
   situación para interpretar la regularización.

👉 Conclusión provisional:
   La regularización funciona, pero todavía nos falta hacer comparables las escalas.
""")

# ============================================================
# 4B. REGULARIZACIÓN CON ESTANDARIZACIÓN
# ============================================================

modelos_estandarizados = {
    "Sin regularizar": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge (L2)": make_pipeline(StandardScaler(), Ridge(alpha=10)),
    "Lasso (L1)": make_pipeline(StandardScaler(), Lasso(alpha=2, max_iter=10000)),
}

coef_estandarizados = pd.DataFrame(index=X_reg.columns)

for nombre, modelo in modelos_estandarizados.items():
    modelo.fit(X_reg_train, y_reg_train)
    estimador = modelo.steps[-1][1]
    coef_estandarizados[nombre] = estimador.coef_.round(3)

print("\n" + "="*70)
print("4B. Coeficientes CON estandarización:\n")
display(coef_estandarizados)

print("""
🧭 ¿Qué mirar ahora?

1. Las variables están en una escala comparable.
   Eso hace mucho más razonable comparar el tamaño de los coeficientes.

2. Ridge (L2):
   - reduce los coeficientes;
   - normalmente conserva todas las variables;
   - sirve para controlar la complejidad sin eliminar variables por completo.

3. Lasso (L1):
   - también reduce coeficientes;
   - puede llevar algunos exactamente a cero;
   - por eso puede funcionar como una forma de selección de variables.

4. Mira nuevamente 'ruido_aleatorio':
   si Lasso la lleva a 0, está mostrando de forma muy clara que esa variable
   no aporta lo suficiente como para justificar mantenerla activa.

⚠️ Importante:
Que Lasso lleve una variable a cero depende del valor de alpha y de los datos.
No es una garantía automática.

✅ Idea que queremos llevarnos:
Cuando usamos regularización, estandarizar ayuda a que la penalización trate
las variables de una manera más comparable e interpretable.
""")

4A. Coeficientes SIN estandarizar:



,Sin regularizar,Ridge (L2),Lasso (L1)
superficie_m2,2.182,2.182,2.180
n_habitaciones,8.194,8.011,7.216
antiguedad_años,-0.860,-0.858,-0.838
ruido_aleatorio,-0.431,-0.393,-0.000



🧭 ¿Qué mirar aquí?

1. Los coeficientes NO están en una escala común.
   Por eso, un coeficiente numéricamente grande no significa necesariamente
   que esa variable sea "más importante".

2. Ridge y Lasso ya están aplicando una penalización, pero su efecto puede ser
   difícil de comparar porque cada variable está medida en unidades distintas.

3. Mira especialmente la fila 'ruido_aleatorio'.
   Puede tener un coeficiente pequeño, pero todavía no estamos en la mejor
   situación para interpretar la regularización.

👉 Conclusión provisional:
   La regularización funciona, pero todavía nos falta hacer comparables las escalas.


4B. Coeficientes CON estandarización:



,Sin regularizar,Ridge (L2),Lasso (L1)
superficie_m2,56.402,53.801,54.311
n_habitaciones,11.942,11.379,9.708
antiguedad_años,-12.082,-11.309,-9.694
ruido_aleatorio,-0.400,-0.435,-0.000



🧭 ¿Qué mirar ahora?

1. Las variables están en una escala comparable.
   Eso hace mucho más razonable comparar el tamaño de los coeficientes.

2. Ridge (L2):
   - reduce los coeficientes;
   - normalmente conserva todas las variables;
   - sirve para controlar la complejidad sin eliminar variables por completo.

3. Lasso (L1):
   - también reduce coeficientes;
   - puede llevar algunos exactamente a cero;
   - por eso puede funcionar como una forma de selección de variables.

4. Mira nuevamente 'ruido_aleatorio':
   si Lasso la lleva a 0, está mostrando de forma muy clara que esa variable
   no aporta lo suficiente como para justificar mantenerla activa.

⚠️ Importante:
Que Lasso lleve una variable a cero depende del valor de alpha y de los datos.
No es una garantía automática.

✅ Idea que queremos llevarnos:
Cuando usamos regularización, estandarizar ayuda a que la penalización trate
las variables de una manera más comparable e interpretable.



---
## **5️⃣ Bloque 5 — Validación cruzada (K-Fold)**

Una sola división entre entrenamiento y validación puede depender bastante de **qué observaciones quedaron en cada lado**.

La **validación cruzada de K pliegues** repite la evaluación varias veces:

1. divide los datos en K partes;
2. en cada iteración usa una parte como **validación** y las restantes para entrenamiento;
3. repite hasta que cada parte haya sido usada una vez como validación;
4. promedia los resultados.

### ¿Por qué nos sirve?
Porque reduce el riesgo de sacar una conclusión a partir de una única partición que, por casualidad, haya resultado especialmente fácil o difícil.

> **Idea clave:** K-Fold no reemplaza al conjunto de prueba final.  
> K-Fold nos ayuda **durante el desarrollo** del modelo.  
> El conjunto de prueba sigue reservado para la evaluación final.

### Qué mirar en los resultados
No solo importa el **promedio** del desempeño, sino también cuánto cambia entre pliegues.

- Si los resultados son parecidos entre sí, el modelo se comporta de forma relativamente estable.
- Si cambian mucho, puede ser una señal de que el resultado depende bastante de la muestra que tocó en cada partición.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

modelo_cv = make_pipeline(StandardScaler(), Ridge(alpha=10))

puntajes = cross_val_score(modelo_cv, X, y, cv=kf, scoring="r2")

print(f"R² obtenido en cada uno de los 5 pliegues de validación: {puntajes.round(3)}")
print(f"\n✅ R² promedio: {puntajes.mean():.3f}")
print(f"✅ Desviación entre pliegues: {puntajes.std():.3f}")

print("""
🧭 ¿Cómo interpretar estos resultados?

- El R² promedio resume el desempeño del modelo a través de varias particiones.
- La desviación nos dice cuánto cambió el resultado entre un pliegue y otro.
- Una desviación pequeña sugiere un comportamiento más estable.
- Una desviación grande indica que el desempeño depende bastante de qué datos
  quedaron para entrenar y cuáles para validar.

⚠️ Recuerda:
Esto sigue siendo VALIDACIÓN.
El conjunto de PRUEBA final todavía no se ha tocado.
""")

R² obtenido en cada uno de los 5 pliegues de validación: [0.905 0.928 0.919 0.92  0.934]

✅ R² promedio: 0.921
✅ Desviación entre pliegues: 0.010

🧭 ¿Cómo interpretar estos resultados?

- El R² promedio resume el desempeño del modelo a través de varias particiones.
- La desviación nos dice cuánto cambió el resultado entre un pliegue y otro.
- Una desviación pequeña sugiere un comportamiento más estable.
- Una desviación grande indica que el desempeño depende bastante de qué datos
  quedaron para entrenar y cuáles para validar.

⚠️ Recuerda:
Esto sigue siendo VALIDACIÓN.
El conjunto de PRUEBA final todavía no se ha tocado.



---
## **6️⃣ Bloque 6 — Muestreo estratificado: cuando las clases están desbalanceadas**

Ahora pasamos a un problema de clasificación.

Supongamos que queremos detectar fraude y que solo un **5%** de los casos corresponde a fraude. Eso significa que tenemos una clase muy poco frecuente.

Si dividimos los datos al azar sin ningún cuidado, podría ocurrir que una de las particiones quede con muy pocos casos de fraude.

### ¿Qué hace `stratify`?
Hace que entrenamiento y prueba conserven aproximadamente la misma proporción de cada clase que tenía el dataset original.

> **Idea clave:** estratificar no “arregla” el desbalance.  
> Lo que hace es **preservarlo de manera controlada** al dividir los datos.

### Qué mirar en el resultado
Compara la proporción de fraude:
- en el dataset completo;
- en entrenamiento;
- en prueba.

Si las proporciones son muy similares, la estratificación está haciendo su trabajo.

In [ ]:
X_fraude, y_fraude = make_classification(
    n_samples=2000, n_features=5, weights=[0.95, 0.05], random_state=42
)

Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_fraude, y_fraude,
    test_size=0.25,
    stratify=y_fraude,
    random_state=42
)

prop_total = y_fraude.mean()
prop_train = yf_train.mean()
prop_test = yf_test.mean()

print("📌 Proporción de casos positivos (fraude)")
print(f"Dataset completo: {prop_total:.3%}")
print(f"Entrenamiento:    {prop_train:.3%}")
print(f"Test:           {prop_test:.3%}")

print("""
🧭 ¿Cómo interpretarlo?

- Las tres proporciones deberían ser muy parecidas.
- Eso significa que entrenamiento y prueba conservan la estructura del problema original.
- Si la clase positiva es rara, esto evita que una partición quede accidentalmente
  con muy pocos casos positivos.

⚠️ Ojo:
Estratificar NO aumenta la cantidad de fraudes.
Solo asegura que la proporción se mantenga al dividir.
""")

📌 Proporción de casos positivos (fraude)
Dataset completo: 5.450%
Entrenamiento:    5.467%
Test:           5.400%

🧭 ¿Cómo interpretarlo?

- Las tres proporciones deberían ser muy parecidas.
- Eso significa que entrenamiento y prueba conservan la estructura del problema original.
- Si la clase positiva es rara, esto evita que una partición quede accidentalmente
  con muy pocos casos positivos.

⚠️ Ojo:
Estratificar NO aumenta la cantidad de fraudes.
Solo asegura que la proporción se mantenga al dividir.



---
## **7️⃣ Bloque 7 — Métricas: ¿cómo sabemos si el modelo aprendió bien?**

En clasificación, una sola métrica puede contar una historia incompleta.

Si el 95% de los casos son “no fraude”, un modelo que siempre prediga “no fraude” puede obtener cerca de **95% de Accuracy** y aun así ser completamente inútil para detectar fraude.

Por eso miramos varias métricas:

- **Accuracy**: de todas las predicciones, ¿qué proporción fue correcta?
- **Precision**: de todo lo que el modelo marcó como positivo, ¿qué proporción realmente era positiva?
- **Recall**: de todos los positivos reales, ¿qué proporción logró detectar?
- **F1-Score**: combina Precision y Recall en una sola medida.

### Una forma simple de recordarlas

- **Precision** pregunta: “Cuando digo que hay fraude, ¿cuántas veces tengo razón?”
- **Recall** pregunta: “De todos los fraudes que realmente existían, ¿cuántos encontré?”

> **Idea clave:** no existe una métrica universalmente “mejor”.  
> La métrica importante depende del **costo de equivocarse** en el problema real.

En un problema donde dejar escapar un fraude es muy costoso, normalmente tendremos especial interés en **Recall**.

In [ ]:
# --- Clasificación: fraude ---
modelo_fraude = LogisticRegression(max_iter=1000)
modelo_fraude.fit(Xf_train, yf_train)
pred_fraude = modelo_fraude.predict(Xf_test)

acc = accuracy_score(yf_test, pred_fraude)
prec = precision_score(yf_test, pred_fraude, zero_division=0)
rec = recall_score(yf_test, pred_fraude, zero_division=0)
f1 = f1_score(yf_test, pred_fraude, zero_division=0)

print("📌 MÉTRICAS DE CLASIFICACIÓN (fraude)")
print(f"Accuracy : {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall   : {rec:.3f}")
print(f"F1-Score : {f1:.3f}")

print("""
🧭 ¿Cómo leer estos números?

1. Accuracy
   Resume cuántas predicciones fueron correctas en total.
   Pero en clases desbalanceadas puede verse alto incluso si el modelo
   detecta mal la clase minoritaria.

2. Precision
   Nos dice qué tan confiables son las alertas positivas.
   Precision alta = pocas falsas alarmas.

3. Recall
   Nos dice cuántos de los positivos reales logramos encontrar.
   Recall alto = dejamos escapar pocos casos positivos.

4. F1-Score
   Resume el equilibrio entre Precision y Recall.

👉 No mires una sola métrica.
Pregúntate primero: ¿qué error cuesta más en este negocio?
""")

📌 MÉTRICAS DE CLASIFICACIÓN (fraude)
Accuracy : 0.952
Precision: 0.588
Recall   : 0.370
F1-Score : 0.455

🧭 ¿Cómo leer estos números?

1. Accuracy
   Resume cuántas predicciones fueron correctas en total.
   Pero en clases desbalanceadas puede verse alto incluso si el modelo
   detecta mal la clase minoritaria.

2. Precision
   Nos dice qué tan confiables son las alertas positivas.
   Precision alta = pocas falsas alarmas.

3. Recall
   Nos dice cuántos de los positivos reales logramos encontrar.
   Recall alto = dejamos escapar pocos casos positivos.

4. F1-Score
   Resume el equilibrio entre Precision y Recall.

👉 No mires una sola métrica.
Pregúntate primero: ¿qué error cuesta más en este negocio?



In [ ]:
# --- Regresión: precio de viviendas ---
# Recién aquí usamos el conjunto de PRUEBA reservado en el Bloque 3.

modelo_precio = make_pipeline(StandardScaler(), Ridge(alpha=10))
modelo_precio.fit(X_train, y_train)
pred_precio = modelo_precio.predict(X_test)

mae = mean_absolute_error(y_test, pred_precio)
rmse = mean_squared_error(y_test, pred_precio) ** 0.5
r2 = r2_score(y_test, pred_precio)

print("📌 MÉTRICAS DE REGRESIÓN (precio de viviendas)")
print(f"MAE  : {mae:,.0f} CLP")
print(f"RMSE : {rmse:,.0f} CLP")
print(f"R²   : {r2:.3f}")

print(f"""
🧭 ¿Cómo interpretar estas métricas?

1. MAE
   En promedio, la predicción se aleja del precio real en aproximadamente
   {mae:,.0f} CLP.
   Es una métrica muy intuitiva porque está en la misma unidad del problema.

2. RMSE
   También mide error, pero castiga más los errores grandes.
   Si RMSE es bastante mayor que MAE, puede indicar que existen algunos errores
   especialmente grandes.

3. R²
   Indica qué proporción de la variabilidad de los precios logra explicar el modelo.
   Un valor más cercano a 1 implica mayor capacidad explicativa.

⚠️ Importante:
Un R² alto no significa que el modelo sea "perfecto".
Siempre conviene mirar también el tamaño real de los errores (MAE y RMSE)
y preguntarse si ese nivel de error es aceptable para el negocio.
""")

📌 MÉTRICAS DE REGRESIÓN (precio de viviendas)
MAE  : 13,866,828 CLP
RMSE : 16,152,425 CLP
R²   : 0.904

🧭 ¿Cómo interpretar estas métricas?

1. MAE
   En promedio, la predicción se aleja del precio real en aproximadamente
   13,866,828 CLP.
   Es una métrica muy intuitiva porque está en la misma unidad del problema.

2. RMSE
   También mide error, pero castiga más los errores grandes.
   Si RMSE es bastante mayor que MAE, puede indicar que existen algunos errores
   especialmente grandes.

3. R²
   Indica qué proporción de la variabilidad de los precios logra explicar el modelo.
   Un valor más cercano a 1 implica mayor capacidad explicativa.

⚠️ Importante:
Un R² alto no significa que el modelo sea "perfecto".
Siempre conviene mirar también el tamaño real de los errores (MAE y RMSE)
y preguntarse si ese nivel de error es aceptable para el negocio.



---
## **✅ Cierre de la sesión**

Hoy vimos varias **reglas del juego** que ayudan a desarrollar y evaluar modelos de Machine Learning de manera confiable:

1. Reconocer los **tipos de variables**.
2. Cuidar la **calidad de los datos**.
3. Separar datos en **entrenamiento / validación / prueba**.
4. Entender por qué la **escala de las variables** importa al regularizar.
5. Usar **Ridge y Lasso** para controlar la complejidad.
6. Usar **validación cruzada** para evaluar con mayor estabilidad durante el desarrollo.
7. **Estratificar** cuando las clases están desbalanceadas.
8. Elegir **métricas acordes al problema y al costo de los errores**.

### Si solo nos llevamos cuatro ideas, que sean estas:

- **No evaluamos con los mismos datos con los que entrenamos.**
- **Regularización y estandarización están estrechamente relacionadas.**
- **Accuracy puede engañar cuando las clases están desbalanceadas.**
- **La mejor métrica depende de qué error es más costoso en el negocio.**

> *"Si entendemos las piezas, podremos armar el engranaje."*